<a href="https://colab.research.google.com/github/babynaz/Gold_Price_Prediction/blob/main/Copy_of_Gold_Price_Prediction_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Part 1 : Load & Merge Data

In [1]:
# Import Libraries

# Core
import pandas as pd
import numpy as np

# ML
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

# Deep Learning
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, LSTM, Dropout, Input
from tensorflow.keras.layers import RepeatVector, TimeDistributed

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

In [2]:
# Load datasets

gold = pd.read_csv("gold_2015_2025.csv")
oil  = pd.read_csv("oil_2015_2025.csv")
usd  = pd.read_csv("USD_2015_2025.csv")
dji  = pd.read_csv("dji_2015_2025.csv")
cop  = pd.read_csv("copper_2015_2025.csv")
nas  = pd.read_csv("nasdaq_daily_2015_2025.csv")
sp  = pd.read_csv("S_P_2015_2025-2.csv")
btc = pd.read_csv("btc_2015_2025.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'gold_2015_2025.csv'

In [ ]:
# Convert to datetime
gold['datetime'] = pd.to_datetime(gold['datetime'])
oil['datetime']  = pd.to_datetime(oil['datetime'])
usd['datetime']  = pd.to_datetime(usd['datetime'])
dji['datetime']  = pd.to_datetime(dji['datetime'])
cop['datetime']  = pd.to_datetime(cop['datetime'])
nas['datetime']  = pd.to_datetime(nas['datetime'])
sp['datetime']  = pd.to_datetime(sp['datetime'])
btc['datetime']  = pd.to_datetime(btc['datetime'])

In [ ]:
# Gold goes back to 1800 (Keep only matching years)
gold = gold[gold['datetime'] >= "2015-01-01"]

In [ ]:
# Convert timestamp → date only
gold['date'] = gold['datetime'].dt.date
oil['date']  = oil['datetime'].dt.date
usd['date']  = usd['datetime'].dt.date
dji['date']  = dji['datetime'].dt.date
cop['date']  = cop['datetime'].dt.date
nas['date']  = nas['datetime'].dt.date
sp['date']  = sp['datetime'].dt.date
btc['date']  = btc['datetime'].dt.date

In [ ]:
# Merge all

dfs_to_merge = {
    "gold": gold,
    "usd": usd,
    "dji": dji,
    "cop": cop,
    "nas": nas,
    "sp": sp,
    "oil": oil,
    "btc": btc
}

df = gold.copy()

for name, current_df in dfs_to_merge.items():
    if not current_df.empty:
        if name != "gold": # gold is the base
            df = df.merge(current_df, on="date", how="inner", suffixes=("", f"_{name}"))
    else:
        print(f"Warning: '{name}' dataframe is empty and will not be merged.")


In [ ]:
df

# Part 3 : Define features

In [ ]:
# PART 2 : Feature Engineering

# Create Lag Features
df['lag1'] = df['close'].shift(1)
df['lag5'] = df['close'].shift(5)
df['lag10'] = df['close'].shift(10)

# Moving Averages
df['MA5'] = df['close'].rolling(5).mean()
df['MA20'] = df['close'].rolling(20).mean()
df['MA50'] = df['close'].rolling(50).mean()

# Volatility + Returns
df['volatility'] = df['close'].rolling(window=10).std()
df['return'] = df['close'].pct_change()

# Targets
df['y_t1_price'] = df['close'].shift(-1)
df['y_t1_dir']   = (df['close'].shift(-1) > df['close']).astype(int)

# Clean NaN rows
df = df.dropna().reset_index(drop=True)

# PART 3 : Define features
features = ['close', 'close_usd','close_dji','close_cop','close_oil','close_nas','close_sp','close_btc']

X = df[features]
y_reg = df['y_t1_price']
y_cls = df['y_t1_dir']

In [ ]:
# Use 80% for training, 20% for testing
train_size = int(len(X) * 0.8)

X_train = X.iloc[:train_size]
X_test  = X.iloc[train_size:]

y_reg_train = y_reg.iloc[:train_size]
y_reg_test  = y_reg.iloc[train_size:]



In [ ]:
seq_length = 30

def make_sequence(data, target):
    Xs, ys = [], []
    for i in range(len(data) - seq_length):
        Xs.append(data.iloc[i:i+seq_length].values)
        ys.append(target.iloc[i+seq_length])
    return np.array(Xs), np.array(ys)

X_lstm_reg_train, y_lstm_reg_train = make_sequence(X_train, y_reg_train)
X_lstm_reg_test,  y_lstm_reg_test  = make_sequence(X_test, y_reg_test)



print(X_lstm_reg_train.shape)
print(X_lstm_reg_test.shape)


# PART 4 : Random Forest (Traditional ML)
Predict both:
*   t+1 price
*   t+1 direction

In [ ]:
# (Regression)

from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor

param_dist = {
    'n_estimators': [300, 500, 800],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None]
}

rf = RandomForestRegressor(random_state=42)

random_search = RandomizedSearchCV(
    rf, param_distributions=param_dist,
    n_iter=20, cv=5, scoring='neg_mean_squared_error',
    verbose=1, n_jobs=-1
)

random_search.fit(X_train, y_reg_train)
rf_reg = random_search.best_estimator_


pred_rf_reg = rf_reg.predict(X_test)

rf_reg_mae = mean_absolute_error(y_reg_test, pred_rf_reg)
rf_reg_rmse = np.sqrt(mean_squared_error(y_reg_test, pred_rf_reg))
rf_reg_r2 = r2_score(y_reg_test, pred_rf_reg)

print("Random Forest (Regression)")
print("MAE :", rf_reg_mae)
print("RMSE:", rf_reg_rmse)
print("R²  :", rf_reg_r2)

# PART 5 : LSTM (Deep Learning)
Uses 30-day window

In [ ]:
# Reshap 3D
# Moved the make_sequence function and its calls to an earlier cell to resolve NameError.

In [ ]:
# Regression
from keras.optimizers import Adam
from keras.callbacks import ReduceLROnPlateau
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.preprocessing import MinMaxScaler

from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout
from keras.optimizers import Adam

reg_lstm = Sequential([
    LSTM(128, return_sequences=True, input_shape=(seq_length, X_train.shape[1])),
    Dropout(0.3),
    LSTM(64),
    Dropout(0.3),
    Dense(32, activation='tanh'),
    Dense(1)
])

reg_lstm.compile(optimizer=Adam(learning_rate=0.001), loss='mse')

# Callbacks
from keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)

# Train
reg_lstm.fit(X_lstm_reg_train, y_lstm_reg_train,
             validation_split=0.2,
             epochs=250,
             batch_size=64,
             callbacks=[early_stop, lr_scheduler],
             verbose=1)




x_scaler = MinMaxScaler()
X_lstm_reg_train = x_scaler.fit_transform(X_lstm_reg_train.reshape(-1, X_lstm_reg_train.shape[-1])).reshape(X_lstm_reg_train.shape)

y_scaler = MinMaxScaler()
y_lstm_reg_train = y_scaler.fit_transform(y_lstm_reg_train.reshape(-1, 1)).flatten()

early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)

reg_lstm.compile(optimizer=Adam(learning_rate=0.0005), loss='mse')



reg_lstm.fit(X_lstm_reg_train, y_lstm_reg_train, callbacks=[early_stop, lr_scheduler],
             epochs=250, batch_size=64 ,verbose=1)

pred_lstm_reg = reg_lstm.predict(X_lstm_reg_test).flatten()

lstm_reg_mae  = mean_absolute_error(y_lstm_reg_test, pred_lstm_reg)
lstm_reg_rmse = np.sqrt(mean_squared_error(y_lstm_reg_test, pred_lstm_reg))
lstm_reg_r2   = r2_score(y_lstm_reg_test, pred_lstm_reg)

In [ ]:
print("LSTM Regression MAE :", lstm_reg_mae)
print("LSTM Regression RMSE:", lstm_reg_rmse)
print("LSTM Regression R2  :", lstm_reg_r2)

# PART 6 : Autoencoder + MLP (Transfer Learning-Style Model)
* Deep method
* Feature extraction layer
* Transfer-learning-like architecture
(We learn compressed features → use them to predict gold price)

In [ ]:
# (Regression)

from keras.models import Model
from keras.layers import Input, Dense

input_dim = X_train.shape[1]
encoding_dim = 8  # increase from 4 to 8 or 16

# Encoder
input_layer = Input(shape=(input_dim,))
encoded = Dense(64, activation='relu')(input_layer)
encoded = Dense(32, activation='relu')(encoded)
encoded_output = Dense(encoding_dim, activation='relu')(encoded)

# Decoder
decoded = Dense(32, activation='relu')(encoded_output)
decoded = Dense(64, activation='relu')(decoded)
decoded_output = Dense(input_dim, activation='linear')(decoded)

autoencoder = Model(input_layer, decoded_output)
encoder = Model(input_layer, encoded_output)

autoencoder.compile(optimizer='adam', loss='mse')


autoencoder.fit(X_train, X_train,
                epochs=400,
                batch_size=64,
                validation_split=0.2,
                verbose=1)

In [ ]:
# Extract encoded features
X_train_encoded = encoder.predict(X_train)
X_test_encoded = encoder.predict(X_test)

In [ ]:
# MLP on top of encoded features
from keras.models import Sequential
from keras.layers import Dense, Dropout, BatchNormalization

mlp = Sequential([
    Dense(32, activation='relu', input_shape=(encoding_dim,)),
    BatchNormalization(),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(1)
])

mlp.compile(optimizer='adam', loss='mse')  # try 'huber' for robustness

# Add early stopping & learning rate scheduler
from keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)

# Train with validation split
mlp.fit(X_train_encoded, y_reg_train,
        validation_split=0.2,
        epochs=200,
        batch_size=64,
        callbacks=[early_stop, lr_scheduler],
        verbose=1)


In [ ]:
# Predictions
pred_ae = mlp.predict(X_test_encoded).flatten()

ae_mae  = mean_absolute_error(y_reg_test, pred_ae)
ae_rmse = np.sqrt(mean_squared_error(y_reg_test, pred_ae))
ae_r2   = r2_score(y_reg_test, pred_ae)

In [ ]:
print("Autoencoder+MLP Regression MAE :", ae_mae)
print("Autoencoder+MLP Regression RMSE:", ae_rmse)
print("Autoencoder+MLP Regression R2  :", ae_r2)

# PART 6 : Model Comparison Table

In [ ]:
results = pd.DataFrame({
    "Model": ["RF Regression", "LSTM Regression", "Autoencoder Regression"],
    "MAE":  [rf_reg_mae, lstm_reg_mae, ae_mae],
    "RMSE": [rf_reg_rmse, lstm_reg_rmse, ae_rmse],
    "R2":   [rf_reg_r2, lstm_reg_r2, ae_r2]
})

results

# PART 7 : VISUALIZATIONS
Plot actual vs predicted for each model

In [ ]:
# Random Forest

plt.figure(figsize=(12,5))
plt.plot(y_reg_test.values, label="Actual")
plt.plot(pred_rf_reg, label="RF Predicted")
plt.title("Random Forest — Gold Price Prediction (t+1)")
plt.legend()
plt.show()

In [ ]:
# LSTM

plt.figure(figsize=(12,5))
plt.plot(y_lstm_reg_test, label="Actual")
plt.plot(pred_lstm_reg, label="LSTM Predicted")
plt.title("LSTM — Gold Price Prediction (t+1)")
plt.legend()
plt.show()

In [ ]:
# Autoencoder + MLP

plt.figure(figsize=(12,5))
plt.plot(y_reg_test.values, label="Actual")
plt.plot(pred_ae, label="AE + MLP Predicted")
plt.title("Autoencoder + MLP — Gold Price Prediction (t+1)")
plt.legend()
plt.show()

In [ ]:
# Prediction plots

plt.figure(figsize=(12,5))
plt.plot(y_reg_test.values[:300], label='Actual')
plt.plot(pred_rf_reg[:300], label='RF Prediction')
plt.plot(pred_lstm_reg[:300], label='LSTM Prediction')
plt.plot(pred_ae[:300], label='Autoencoder Prediction')
plt.legend()
plt.title("Gold Price Prediction Comparison")
plt.show()